# QQP — Experiment #4B: BiLSTM + Bidirectional Bahdanau Cross-Attention (WordPiece, Stopword Masking)

**Purpose:** Evaluate the effect of stopword masking during final multihead additive Bahdanau attention pooling, compared with Experiment #4A's PAD-only masking approach. This experiment also introduces dynamic distributed training using PyTorch DistributedDataParallel (DDP), providing reusable training infrastructure for subsequent experiments.

## Architecture and Preprocessing

- **Tokenizer:** Custom-trained, uncased WordPiece tokenizer with a 30K vocabulary and maximum sequence length of 64 tokens.
- **Embeddings:** 100-dimensional, randomly initialized and trainable from scratch. No pretrained GloVe or BERT weights.
- **Encoder:** Shared two-layer bidirectional LSTM with 384 hidden units per direction, producing 768 features per token. Packed sequences prevent PAD tokens from influencing recurrent computation.
- **Cross-attention:** Four-head bidirectional additive Bahdanau cross-attention. Each head uses 64-dimensional query/key scoring projections and 192-dimensional value projections.
- **Residual connection:** Cross-attention outputs are added to their corresponding BiLSTM representations, followed by LayerNorm.
- **Final pooling:** Independent four-head additive Bahdanau attention pooling with position-level stopword masking, producing a 768-dimensional representation for each question.
- **Classifier:** Combines both question representations, absolute difference, elementwise multiplication, and cosine similarity, producing 3,073 features followed by `3073 → 1024 → 256 → 1`.

## Masking Strategy

- **Cross-attention:** PAD-only masking. Stopwords remain involved in cross-question alignment.
- **Final attention pooling:** Excludes PAD tokens and WordPiece positions belonging to original NLTK English stopwords.
- **Position-level masking:** Uses character offsets to associate WordPiece subwords with stopwords in the original question, rather than globally masking vocabulary IDs.
- **Punctuation and UNK:** Remain eligible for attention pooling.
- **All-masked fallback:** Restores PAD-only masking when no eligible tokens remain for final pooling.

Stopwords remain part of the input sequences and continue influencing BiLSTM and cross-attention representations. Only their direct contribution to final attention pooling is restricted.

## Dynamic Distributed Training

The training infrastructure automatically adapts to the available hardware:

| Available GPUs | Training Strategy |
|---|---|
| 0 | CPU |
| 1 | Standard single-GPU training |
| 2 or more | PyTorch DistributedDataParallel (DDP) |

Two batch-size strategies are supported:

- **Fixed global:** Preserves the global batch size and distributes it across participating replicas.
- **Fixed local:** Preserves the batch size per replica and increases the global batch size proportionally to the number of replicas.

Experiment #4B uses the **fixed-global strategy** with a global batch size of 128.

| Devices | Batch Size per Replica | Global Batch Size |
|---|---:|---:|
| CPU | 128 | 128 |
| 1 GPU | 128 | 128 |
| 2 GPUs | 64 | 128 |
| 4 GPUs | 32 | 128 |

When the global batch size is not divisible by the number of available GPUs, the launcher selects the largest compatible number of GPUs.

DDP handles gradient synchronization automatically, while training and validation losses and classification metrics are aggregated globally. Validation samples are partitioned without duplication.

Only the main process (rank 0) handles MLflow logging, checkpoint saving, model registration, and progress reporting. The parent notebook renders graphical `tqdm.notebook` progress bars from rank 0's structured progress updates.

## Notebook Execution

The notebook preserves its interactive data exploration, preprocessing, tokenizer inspection, dataset inspection, and model diagnostics.

Actual training is executed through a temporary `train_worker.py` script, automatically generated from notebook cells marked with `# worker`.

The worker is launched directly through Python for CPU and single-GPU training, or through `torchrun` for distributed multi-GPU training.

The notebook remains the single source of truth. The generated worker script is not committed to the repository or included in the final transfer archive.

## Kaggle Inputs

Attach the versioned private **WordPiece Training Assets** dataset containing:

1. `02b-lstm-bahdanau-crossattn-wordpiece-stopmask-train.ipynb` — clean notebook source without execution outputs.
2. `requirements.py` and the latest MLflow SQLite database snapshot.
3. `artifacts/tokenizers/wordpiece_uncased_30k/` — existing trained WordPiece tokenizer files.
4. `data/processed/train_split.csv` and `data/processed/valid_split.csv` — existing prepared dataset splits.

The official Quora Question Pairs competition dataset may also be attached for competition association. Training continues to use the existing prepared dataset splits.

The assets are copied into `/kaggle/working/qqp`. Neither the tokenizer nor the dataset splits are retrained or regenerated.

## Architecture Export and MLflow

- The model architecture is automatically exported as `02b_stopmask_model_architecture.py`.
- The exact NLTK stopword list is saved as `stopwords.json` for reproducible preprocessing and inference.
- The existing MLflow workflow handles configuration, checkpoints, training history, model registration, and notebook source logging.
- Checkpoints and registered models use the underlying model without the DDP wrapper, preserving compatibility with non-distributed inference.
- The final `qqp-transfer.zip` includes the updated MLflow database, generated artifacts, and the exported architecture Python file.
- The existing Kaggle-to-Windows MLflow database migration and artifact-path restoration are preserved.

### Additional Execution Parameters

| Parameter | Description |
|---|---|
| `training_strategy` | CPU, Single-GPU, or DDP |
| `gpu_count` | Number of participating GPUs |
| `available_gpu_count` | Number of available GPUs |
| `gpu_type` | GPU hardware type |
| `world_size` | Number of participating processes |
| `batch_size_strategy` | Fixed-global or fixed-local |
| `local_batch_size` | Batch size per replica |
| `global_batch_size` | Effective global batch size |
| `optimizer_steps` | Number of optimizer updates |
| `samples_processed` | Total training samples processed |
| `training_time_seconds` | Total training wall-clock duration |
| `samples_per_second` | Global training throughput |
| `peak_gpu_memory_mb` | Maximum PyTorch allocated GPU memory across replicas, measured in MiB |
| `gpu_hours` | Participating GPU count multiplied by training duration in hours |

### Additional MLflow Tags

- `world_size`
- `gpu_hours`
- `batch_size_per_replica`

**Important:** The clean source notebook in the Kaggle assets dataset must match the code used for training, architecture export, and worker-script generation.

## Comparison Protocol

- **Experiment #2:** WordPiece + shared BiLSTM + independent Bahdanau attention pooling, without cross-attention.
- **Experiment #4A:** Bidirectional Bahdanau cross-attention with PAD-only additive attention pooling.
- **Experiment #4B:** Bidirectional Bahdanau cross-attention with position-level stopword-masked additive attention pooling.
- **Experiment #4C:** Bidirectional Bahdanau cross-attention with masked Mean+Max pooling.

The trained tokenizer, dataset splits, model dimensions, loss function, optimizer, learning rate, checkpoint selection, and evaluation protocol are preserved across the Experiment #4 variants wherever applicable.

## Comparison Considerations

Experiment #4B introduces both stopword masking and distributed training infrastructure. Although the fixed-global strategy preserves the effective batch size, distributed sampling, random-number streams, and numerical differences may affect training trajectories.

Training duration alone is not sufficient for comparing architectures trained on different hardware configurations. GPU-hours, throughput, memory consumption, optimizer steps, and hardware information should also be considered.

For strict computational-efficiency comparisons, shortlisted architectures should be benchmarked under identical hardware and execution settings.

**Experiment status:** Prepared for training. Distributed execution and MLflow integration require validation before full training. No completed training results are claimed.

In [1]:
%%capture
%pip install mlflow nltk ipywidgets
%pip install --upgrade torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu126

In [2]:
# worker
from collections import defaultdict
from contextlib import nullcontext
from datetime import datetime, timezone
import inspect
import json
import logging
import os
from pathlib import Path
import random
import re
import shutil
import subprocess
import sys
from time import perf_counter
from tqdm.notebook import tqdm
import warnings
import zipfile

import nltk
from nltk.corpus import stopwords

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_curve

import sqlite3

import torch
from torch import nn, optim
from torch.amp import GradScaler, autocast
import torch.distributed as dist
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)
from torch.optim.lr_scheduler import (
    CosineAnnealingWarmRestarts,
    CosineAnnealingLR,
    OneCycleLR,
    ReduceLROnPlateau,
)
from torch.utils.data import (
    Dataset, 
    DataLoader, 
    DistributedSampler, 
    Subset,
)
from torchmetrics.classification import (
    BinaryAccuracy,
    BinaryPrecision,
    BinaryRecall,
    BinaryF1Score,
    AUROC,
    AveragePrecision,
    PrecisionRecallCurve,
)
from transformers import PreTrainedTokenizerFast

warnings.filterwarnings("ignore", message=".*Pickle or CloudPickle.*")

In [3]:
print("PyTorch Version:", torch.__version__)
print("Available GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU names:", torch.cuda.get_device_name(0))
    print("GPU Architecture:", torch.cuda.get_arch_list())

PyTorch Version: 2.8.0+cu126
Available GPUs: 2
GPU names: Tesla T4
GPU Architecture: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']


In [4]:
INPUT_ROOT = Path(
    "/kaggle/input/datasets/amirmohamadaskari/"
    "quora-question-pairs-wordpiece-training-assets"
)
WORK_ROOT = Path("/kaggle/working/qqp")
WORK_ROOT.mkdir(parents=True, exist_ok=True)


# Copy the existing MLflow database into writable storage
DB_NAME = "mlflow-quora-questions-pairs.db"

SOURCE_DB = INPUT_ROOT / DB_NAME
WORKING_DB = WORK_ROOT / DB_NAME

if not SOURCE_DB.is_file():
    raise FileNotFoundError(
        f"MLflow database not found in Kaggle dataset: {SOURCE_DB}"
    )

if not WORKING_DB.exists():
    shutil.copy2(SOURCE_DB, WORKING_DB)
    print(">>> Existing MLflow database imported successfully!")
else:
    print(">>> Using the existing working MLflow database.")

for folder in [
    "data/processed",
    "artifacts/tokenizers/wordpiece_uncased_30k",
]:
    shutil.copytree(
        INPUT_ROOT / folder,
        WORK_ROOT / folder,
        dirs_exist_ok=True,
    )

for filename in [
    "requirements.py",
    "02b-lstm-bahdanau-crossattn-wordpiece-stopmask-train.ipynb",
]:
    shutil.copy2(
        INPUT_ROOT / filename,
        WORK_ROOT / filename,
    )

os.chdir(WORK_ROOT)

if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

print("Project directory:", Path.cwd())
print("Setup complete.")

>>> Existing MLflow database imported successfully!
Project directory: /kaggle/working/qqp
Setup complete.


In [5]:
%load_ext requirements

In [6]:
# worker
class SystemConfig:
    IS_DETERMINISTIC = False
    _PROJECT_ROOT = Path.cwd()  # repository root
    _DB_PATH = _PROJECT_ROOT / "mlflow-quora-questions-pairs.db"
    MLFLOW_TRACKING_URI = f"sqlite:///{_DB_PATH.as_posix()}"
    NEXT_LINE_COUNTER = 180
    SEED = 28
    USED_SCALER = False
    @classmethod
    def get_device(cls):
        if torch.cuda.is_available():
            torch.cuda.set_device(cls.LOCAL_RANK)
            return torch.device("cuda", cls.LOCAL_RANK)
    
        return torch.device("cpu")
        
    DEVICE = None

    @classmethod
    def configure(cls):
        cls.WORLD_SIZE = int(os.environ.get("WORLD_SIZE", 1))
        cls.RANK = int(os.environ.get("RANK", 0))
        cls.LOCAL_RANK = int(os.environ.get("LOCAL_RANK", 0))
    
        cls.USE_DDP = cls.WORLD_SIZE > 1
        cls.IS_MAIN = cls.RANK == 0
    
        cls.DEVICE = cls.get_device()
    
        if cls.USE_DDP:
            dist.init_process_group(backend="nccl", device_id=cls.DEVICE)
    
        cls.AVAILABLE_GPU_COUNT = torch.cuda.device_count()
    
        cls.GPU_COUNT = (
            cls.WORLD_SIZE if cls.DEVICE.type == "cuda" else 0
        )
    
        cls.GPU_TYPE = (
            torch.cuda.get_device_name(cls.DEVICE)
            if cls.GPU_COUNT else "None"
        )
    
        cls.TRAINING_STRATEGY = (
            "DDP" if cls.USE_DDP
            else "Single-GPU" if cls.GPU_COUNT
            else "CPU"
        )
    
        if cls.IS_MAIN:
            print(f">>> Training strategy: {cls.TRAINING_STRATEGY}")
            print(f">>> GPUs used: {cls.GPU_COUNT}")
            print(f">>> GPU type: {cls.GPU_TYPE}")
    
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [7]:
# worker
class PathConfig:
    ROOT_DIR = Path().cwd()

    # -------------------------
    # Data
    # -------------------------
    DATA_DIR = ROOT_DIR / "data"
    RAW_DATA_DIR = DATA_DIR / "raw"
    PROCESSED_DATA_DIR = DATA_DIR / "processed"

    TRAIN_CSV_PATH = PROCESSED_DATA_DIR / "train_split.csv"
    VALID_CSV_PATH = PROCESSED_DATA_DIR / "valid_split.csv"

    # -------------------------
    # Artifacts
    # -------------------------
    ARTIFACT_DIR = ROOT_DIR / "artifacts"

    # Persistent shared tokenizer artifact
    TOKENIZER_DIR = (
        ARTIFACT_DIR
        / "tokenizers"
        / "wordpiece_uncased_30k"
    )

    # Run-specific LSTM-Attention artifacts
    MODEL_ARTIFACT_DIR = (
        ARTIFACT_DIR
        / "models"
        / "lstm_bahdanau_crossattn_wordpiece_stopmask"
    )

    CHECKPOINT_DIR = MODEL_ARTIFACT_DIR / "checkpoint"
    CONFIG_PATH = MODEL_ARTIFACT_DIR / "configs.json"
    HISTORY_PATH = MODEL_ARTIFACT_DIR / "training_history.json"
    LABEL_MAPPING_PATH = MODEL_ARTIFACT_DIR / "label_mapping.json"

    # -------------------------
    # Project files
    # -------------------------
    REQ_TEXT = ROOT_DIR / "requirements.txt"
    REQ_SCRIPT = ROOT_DIR / "requirements.py"
    NOTEBOOK_PATH = (
        ROOT_DIR
        / "02b-lstm-bahdanau-crossattn-wordpiece-stopmask-train.ipynb"
    )
    
    MODEL_SCRIPT = ROOT_DIR / (
        f"{NOTEBOOK_PATH.stem.split('-')[0]}_stopmask_model_architecture.py"
    )

    STOPWORDS_PATH = MODEL_ARTIFACT_DIR / "stopwords.json"
    # -------------------------
    # MLflow
    # -------------------------
    MLFLOW_DIR = ROOT_DIR / "mlruns"

    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v
            for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)
            and not isinstance(v, (classmethod, staticmethod))
        }

    @classmethod
    def update_requirements(cls):
        from IPython import get_ipython

        ipython = get_ipython()
        if ipython:
            ipython.run_line_magic(
                "updatereqs",
                str(cls.NOTEBOOK_PATH)
            )

In [8]:
# worker
class TokenConfig:
    PAD_TOKEN = '[PAD]'
    UNK_TOKEN = '[UNK]'
    PAD_IDX = 0
    UNK_IDX = 1
    MAX_LENGTH = 64
    VOCAB_SIZE = 30000
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [9]:
# worker
class LoaderConfig:
    BATCH_SIZE = 128
    NUM_WORKERS = 0
    IS_PIN_MEMORY = True
    BATCH_SIZE_STRATEGY = "fixed_global"
    
    @classmethod
    def configure(cls, world_size):
        if cls.BATCH_SIZE_STRATEGY == "fixed_global":
            if cls.BATCH_SIZE % world_size != 0:
                raise ValueError(
                    "Batch size must be divisible by world size."
                )
    
            cls.LOCAL_BATCH_SIZE = cls.BATCH_SIZE // world_size
            cls.GLOBAL_BATCH_SIZE = cls.BATCH_SIZE
    
        elif cls.BATCH_SIZE_STRATEGY == "fixed_local":
            cls.LOCAL_BATCH_SIZE = cls.BATCH_SIZE
            cls.GLOBAL_BATCH_SIZE = cls.BATCH_SIZE * world_size
    
        else:
            raise ValueError("Invalid batch size strategy.")
    
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [10]:
# worker
class ModelConfig:
    MODEL_TYPE = "LSTM_attention"
    ATTENTION_TYPE = "MultiHead-Bahdanau-Cross-AdditivePool-StopMask"
    TOKENIZER_TYPE = "Wordpiece"
    # Embedding
    LAYER_NORM_EMB = False
    EMBEDDING_TYPE = "Trainable-from-scratch"
    EMB_DIM = 100
    EMB_DP = 0.2
    # Model
    LOSS = "BCE with Logits"
    NUM_HEADS = 4
    ATTENTION_SCORE_DIM = 64
    BIDIRECTIONAL = True
    DROPOUT = 0.35
    HIDDEN_DIM = 384
    LSTM_OUT = HIDDEN_DIM*(2 if BIDIRECTIONAL else 1)
    ATTENTION_DROPOUT = 0.0
    LAYER_NORM_LSTM = False
    LAYER_NORM_ATTENTION = False
    ATTENTION_PROJECTION = False
    if ATTENTION_PROJECTION:
        PROJECT_DIM = HIDDEN_DIM // 2
    ENC_DIM = PROJECT_DIM if ATTENTION_PROJECTION else LSTM_OUT
    if LOSS == "Contrastive Loss":
        MARGIN = 1.0
    elif LOSS == "BCE with Logits":
        LABEL_SMOOTHING = 0.05
        FC_DIMS = [1024, 256]
        FC_DP = 0.4
        SIAMESE_SIMILARITY_PARM = ["Encoded Q1", "Encoded Q2", "Multiplication Q1, Q2", "Abs Subtract Q1, Q2", "Cosine Similarity"]
        MULTIPLE_FC_PARAM = sum(1 for param in SIAMESE_SIMILARITY_PARM
                     if "Q1" in param or "Q2" in param)
        INPUT_FC_DIM = MULTIPLE_FC_PARAM * ENC_DIM
        if any("Cosine" in param for param in SIAMESE_SIMILARITY_PARM):
            INPUT_FC_DIM += 1
    MASK_FILL_NUM = -1e10
    NUM_LAYERS = 2
    SKIP_CONNECTION = True
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [11]:
# worker
class TrainConfig:
    LOSS = ModelConfig.LOSS
    CLIP_NORM = 1.5
    EARLY_STOP_METRIC = "loss"
    CHECKPOINT_METRIC = "F1Score"
    if CHECKPOINT_METRIC == "loss":
        CHECKPOINT_MODE = "min"
    else:
        CHECKPOINT_MODE = "max"
    SCHEDULER_METRIC = "loss"
    EARLY_STOP_MIN_DELTA = 1e-4
    if EARLY_STOP_METRIC in ["F1Score", "Accuracy", "Precision", "Recall"]:
        EARLY_STOP_MODE = "max"
    elif EARLY_STOP_METRIC == "loss":
        EARLY_STOP_MODE = "min"
    EARLY_STOP_PATIENCE = 5
    EPOCHS = 50
    LEARNING_RATE = 3e-4
    METRICS_THRESHOLD = 0.5
    SCHEDULER_TYPE = "ReduceLROnPlateau"
    if SCHEDULER_TYPE == "ReduceLROnPlateau":
        SCHEDULER_FACTOR = 0.5
        SCHEDULER_MIN_LR = 1e-7
        SCHEDULER_PATIENCE = 2
        SCHEDULER_THRESHOLD = 0.01
        SCHEDULER_THRESHOLD_MODE = "rel"
        if SCHEDULER_METRIC in ["F1Score", "Accuracy", "Precision", "Recall"]:
            SCHEDULER_MODE = "max"
        elif SCHEDULER_METRIC == "loss":
            SCHEDULER_MODE = "min"
    elif SCHEDULER_TYPE == "CosineAnnealing":
        SCHEDULER_ETA_MIN = 1e-6
    elif SCHEDULER_TYPE == "CosineAnnealingWarmRestarts":
        SCHEDULER_T_0 = 5
        SCHEDULER_T_MUT = 2
        SCHEDULER_ETA_MIN = 1e-6
    elif SCHEDULER_TYPE == "OneCycleLR":
        SCHEDULER_PCT_START = 0.3
        SCHEDULER_DIV_FACTOR = 25
        SCHEDULER_FINAL_DIV_FACTOR = 1000

    WEIGHT_DECAY = 1e-3
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [12]:
# worker
class PostProcessingConfig:
    INFERENCE_THRESHOLD = 0.5
    METRICS_THRESHOLD = 0.5
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [13]:
# worker
def seed_everything(seed, deterministic=False):
    '''
    Ensures absolute reproducibility.
    '''
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    if deterministic:
        # Only use these for the final "Gold" run to ensure exact results
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        if getattr(sys_cfg, "IS_MAIN", True):
            print(">>> Using STRICT Deterministic mode (Slower).")
    else:
        # Benchmark=True allows cuDNN to find the fastest kernels for your hardware
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
        if getattr(sys_cfg, "IS_MAIN", True):
            print(">>> Using PROTOTYPING mode (Faster).")
    if getattr(sys_cfg, "IS_MAIN", True):
        print(f">>> For Reproducibility, Everything seeded with {seed}!")



In [14]:
# worker
def set_scaler(config=SystemConfig):
    if config.USED_SCALER and sys_cfg.DEVICE.type == "cuda":
        scaler = GradScaler(device=sys_cfg.DEVICE.type, enabled=(sys_cfg.DEVICE.type == 'cuda'))
        if getattr(sys_cfg, "IS_MAIN", True):
            print(">>> Scaler used in training")
    else:
        scaler = None
        if getattr(sys_cfg, "IS_MAIN", True):
            print(">>> Scaler is not Used")

    return scaler

In [15]:
def clean_artifact_directory(artifact_dir: Path):
    if artifact_dir.exists():
        shutil.rmtree(artifact_dir)
        print(f">>> Cleaned local artifact directory: {artifact_dir}")
    artifact_dir.mkdir(parents=True, exist_ok=True)

In [16]:
# worker
sys_cfg = SystemConfig()
path_cfg = PathConfig()
token_cfg = TokenConfig()
train_cfg = TrainConfig()
loader_cfg = LoaderConfig()
model_cfg = ModelConfig()
postprc_cfg = PostProcessingConfig()

In [17]:
print(">>> All Configs are set successfully!")
path_cfg.update_requirements()

>>> All Configs are set successfully!
>>> Scanning: 02b-lstm-bahdanau-crossattn-wordpiece-stopmask-train.ipynb
>>> Detected third-party imports: ['IPython', 'matplotlib', 'mlflow', 'nltk', 'numpy', 'pandas', 'sklearn', 'torch', 'torchmetrics', 'tqdm', 'transformers']
>>> Added: mlflow-tracing, ipython, torchmetrics, torch, pandas, numpy, tqdm, nltk, transformers, matplotlib, scikit-learn
>>> Total packages in requirements.txt: 11


In [18]:
clean_artifact_directory(path_cfg.MODEL_ARTIFACT_DIR)

In [19]:
nltk.download("stopwords", quiet=True, raise_on_error=True)

stop_words = stopwords.words("english")

path_cfg.STOPWORDS_PATH.write_text(
    json.dumps(sorted(stop_words), indent=2),
    encoding="utf-8"
)

2040

In [20]:
# worker
def get_serializable_configs(configs_dict):
    """Create a JSON-serializable version of configs"""
    serializable = {}
    for section, params in configs_dict.items():
        if section not in ["system", "path"]:
            serializable[section] = params
            continue
            
        serializable[section] = {}
        
        for key, value in params.items():
            if isinstance(value, torch.device):
                # Convert torch.device to string
                serializable[section][key] = str(value)
            elif isinstance(value, Path):
                # Convert Path to string
                serializable[section][key] = str(value)
            elif key == "device":
                # Skip or convert device
                serializable[section][key] = str(value) if value.type == "cuda" else "cpu"
            else:
                serializable[section][key] = value
    
    return serializable

In [21]:
# worker
def configs_dict(config_path):
    configs = {}
    configs_names = [
        SystemConfig, PathConfig, TokenConfig, LoaderConfig, TrainConfig, ModelConfig
    ]
    for config in configs_names:
        cfg_clean = f"{config.__name__.replace("Config", "").lower()}"
        configs[cfg_clean] = config.to_dict()

    serializable_configs = get_serializable_configs(configs)
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(serializable_configs, f, ensure_ascii=False, indent=2)
    print(">>> Configs Saved as JSON File!")
    return configs

In [22]:
train_df = pd.read_csv(path_cfg.TRAIN_CSV_PATH)
valid_df = pd.read_csv(path_cfg.VALID_CSV_PATH)

train_df.head()

,question1,question2,is_duplicate
0,What is maximum steering wheel torque in truck?,What is a steering wheel's torque?,0
1,How much does it cost to build a website in In...,How much does it cost to build an ecommerce st...,0
2,Does time stop ever?,Can time stop?,1
3,Did matter exist before the big bang?,What happened before the Big Bang happened?,1
4,If I have a master's in operations research (a...,I am an international masters student with lit...,0


In [23]:
print(f"Number of all pairs in research: {len(train_df) + len(valid_df)}")

Number of all pairs in research: 404287


In [24]:
print("Train missing values:")
display(train_df[["question1", "question2", "is_duplicate"]].isna().sum())

print("Validation missing values:")
display(valid_df[["question1", "question2", "is_duplicate"]].isna().sum())

Train missing values:


question1       0
question2       1
is_duplicate    0
dtype: int64

Validation missing values:


question1       0
question2       0
is_duplicate    0
dtype: int64

In [25]:
train_df["is_duplicate"].value_counts().sort_index()

is_duplicate
0    229521
1    134337
Name: count, dtype: int64

In [26]:
not_dupl = (train_df["is_duplicate"] == 0).sum()
is_dupl = (train_df["is_duplicate"] == 1).sum()
total = len(train_df)
print(
    f"Is duplicate in % : {is_dupl / total * 100}\n"
    f"not duplicate in % : {not_dupl / total * 100}"
)

Is duplicate in % : 36.920172155071484
not duplicate in % : 63.079827844928516


In [27]:
# worker
def preprocess_df(df):
    df = df.copy()

    df['question1'] = df['question1'].fillna('')
    df['question2'] = df['question2'].fillna('')

    before = len(df)
    df = df[
        (df["question1"].str.strip() != "") &
        (df["question2"].str.strip() != "")
    ].reset_index(drop=True)
    after = len(df)
    if getattr(sys_cfg, "IS_MAIN", True):
        print(f">>> Preprocessing complete! Removed empty rows: {before - after}")

    return df

In [28]:
train_df_preprc = preprocess_df(train_df)
valid_df_preprc = preprocess_df(valid_df)

>>> Preprocessing complete! Removed empty rows: 1
>>> Preprocessing complete! Removed empty rows: 0


In [29]:
# worker
class QuoraTokenizer:
    def __init__(self, tokenizer_dir, stop_words, config=token_cfg):
        self.config = config

        self.tokenizer = PreTrainedTokenizerFast.from_pretrained(
            tokenizer_dir,
            local_files_only=True
        )

        self.vocab_size = len(self.tokenizer)
        self.pad_idx = self.tokenizer.pad_token_id
        self.unk_idx = self.tokenizer.unk_token_id
        self.stop_words = set(stop_words)

        if getattr(sys_cfg, "IS_MAIN", True):
            print(">>> WordPiece tokenizer loaded!")
            print(f">>> Vocabulary size: {self.vocab_size:,}")
            print(f">>> PAD token: {self.tokenizer.pad_token} ({self.pad_idx})")
            print(f">>> UNK token: {self.tokenizer.unk_token} ({self.unk_idx})")

    def build_pooling_mask(self, text, encoding):
        stopword_spans = [
            match.span()
            for match in re.finditer(r"\b\w+(?:['’]\w+)*\b", text)
            if match.group().lower().replace("’", "'") in self.stop_words
        ]
    
        mask = []
    
        for idx, (start, end) in zip(
            encoding["input_ids"],
            encoding["offset_mapping"]
        ):
            keep = idx != self.pad_idx
    
            if idx != self.unk_idx and re.search(r"\w", text[start:end]):
                if any(
                    start < word_end and end > word_start
                    for word_start, word_end in stopword_spans
                ):
                    keep = False
    
            mask.append(keep)
    
        return mask

    def encode(self, text, return_pool_mask=False):
        encoding = self.tokenizer(
            text,
            add_special_tokens=False,
            truncation=True,
            padding="max_length",
            max_length=self.config.MAX_LENGTH,
            return_offsets_mapping=return_pool_mask
        )

        input_ids = encoding["input_ids"]
        if return_pool_mask:
            pool_mask = self.build_pooling_mask(text, encoding)
            return input_ids, pool_mask

        return input_ids

    def decode(self, ids, remove_pad=True):
        if remove_pad:
            ids = [
                idx for idx in ids
                if idx != self.pad_idx
            ]

        return self.tokenizer.decode(
            ids,
            skip_special_tokens=True
        )

    def save_label_mapping(self, path):
        path = Path(path)

        label_mapping = {
            "0": "different",
            "1": "duplicated"
        }

        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                label_mapping,
                f,
                ensure_ascii=False,
                indent=2
            )

        print(">>> Label mapping saved!")

In [30]:
tokenizer = QuoraTokenizer(
    tokenizer_dir=path_cfg.TOKENIZER_DIR,
    stop_words=stop_words,
    config=token_cfg
)
print(
    ">>> First 10 tokens:",
    [
        (i, tokenizer.tokenizer.convert_ids_to_tokens(i))
        for i in range(10)
    ]
)

>>> WordPiece tokenizer loaded!
>>> Vocabulary size: 30,000
>>> PAD token: [PAD] (0)
>>> UNK token: [UNK] (1)
>>> First 10 tokens: [(0, '[PAD]'), (1, '[UNK]'), (2, '[CLS]'), (3, '[SEP]'), (4, '[MASK]'), (5, '!'), (6, '"'), (7, '#'), (8, '$'), (9, '%')]


In [31]:
# worker
class QuoraDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.q1 = df["question1"].values
        self.q2 = df["question2"].values
        self.label = df["is_duplicate"].values
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.q1)

    def pos_class_weight(self, device):
        pos_num = (self.label == 1).sum()
        neg_num = (self.label == 0).sum()

        pos_weight = torch.tensor(
            neg_num / pos_num,
            dtype=torch.float32,
            device=device
        )

        return pos_weight

    def __getitem__(self, idx):
        encoded_q1, pool_mask1 = self.tokenizer.encode(
            self.q1[idx], return_pool_mask=True
        )
        encoded_q2, pool_mask2 = self.tokenizer.encode(
            self.q2[idx], return_pool_mask=True
        )

        label = self.label[idx]

        return {
            "q1": torch.tensor(encoded_q1, dtype=torch.long),
            "q2": torch.tensor(encoded_q2, dtype=torch.long),
            "pool_mask1": torch.tensor(pool_mask1, dtype=torch.bool),
            "pool_mask2": torch.tensor(pool_mask2, dtype=torch.bool),
            "label": torch.tensor(label, dtype=torch.float32)
        }

In [32]:
train_dataset = QuoraDataset(train_df_preprc, tokenizer=tokenizer)
val_dataset = QuoraDataset(valid_df_preprc, tokenizer=tokenizer)

In [33]:
for i in range(3):
    item = val_dataset[i]

    print(f"\nSample {i}")

    for j in (1, 2):
        ids = item[f"q{j}"]
        mask = item[f"pool_mask{j}"]

        valid = ids.ne(tokenizer.pad_idx)

        ids = ids[valid].tolist()
        mask = mask[valid].tolist()

        tokens = tokenizer.tokenizer.convert_ids_to_tokens(ids)

        print(f"\nQuestion {j}:")
        print(f"Encoded: {ids}")
        print(f"Decoded: {tokenizer.decode(ids)}")

        display(pd.DataFrame({
            "Token": tokens,
            "Pooling Mask": mask
        }))

    print(f"Label: {item['label'].item()}")
    print("=" * 80)


Sample 0

Question 1:
Encoded: [1481, 51, 2344, 1505, 9432, 43, 10394, 35]
Decoded: can i give my therapist a hug?


,Token,Pooling Mask
0,can,False
1,i,False
2,give,True
3,my,False
4,therapist,True
5,a,False
6,hug,True
7,?,True



Question 2:
Encoded: [1481, 51, 1934, 1505, 9432, 1484, 43, 10394, 35]
Decoded: can i ask my therapist for a hug?


,Token,Pooling Mask
0,can,False
1,i,False
2,ask,True
3,my,False
4,therapist,True
5,for,False
6,a,False
7,hug,True
8,?,True


Label: 0.0

Sample 1

Question 1:
Encoded: [1467, 1462, 51, 2432, 3403, 4248, 1456, 4816, 2317, 2684, 35]
Decoded: how do i run jio 4g in 3g mobile set?


,Token,Pooling Mask
0,how,False
1,do,False
2,i,False
3,run,True
4,jio,True
5,4g,True
6,in,False
7,3g,True
8,mobile,True
9,set,True


... (3 outputs left out)

,Token,Pooling Mask
0,what,False
1,algorithm,True
2,does,False
3,uber,True
4,##pool,True
5,use,True
6,?,True
7,last,True
8,time,True
9,i,False



Question 2:
Encoded: [1467, 1462, 1449, 3307, 9441, 1478, 9710, 3502, 13438, 4865, 1951, 35]
Decoded: how do the uberpool and lyft line matching algorithms compare?


,Token,Pooling Mask
0,how,False
1,do,False
2,the,False
3,uber,True
4,##pool,True
5,and,False
6,lyft,True
7,line,True
8,matching,True
9,algorithms,True


Label: 0.0


In [34]:
# worker
def build_dataloader(dataset, is_train=False, config=loader_cfg):
    sampler = None

    if sys_cfg.USE_DDP:
        if is_train:
            sampler = DistributedSampler(
                dataset,
                shuffle=True,
                seed=sys_cfg.SEED
            )
        else:
            dataset = Subset(
                dataset,
                range(
                    sys_cfg.RANK,
                    len(dataset),
                    sys_cfg.WORLD_SIZE
                )
            )

    return DataLoader(
        dataset,
        batch_size=config.LOCAL_BATCH_SIZE,
        shuffle=is_train and sampler is None,
        sampler=sampler,
        num_workers=config.NUM_WORKERS,
        pin_memory=(
            config.IS_PIN_MEMORY
            and sys_cfg.DEVICE.type == "cuda"
        )
    )

In [35]:
preview_dataloader = DataLoader(
    train_dataset,
    batch_size=loader_cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=loader_cfg.NUM_WORKERS
)

In [36]:
batch = next(iter(preview_dataloader))

print("Batched Question 1 Size:", batch["q1"].shape)
print("Batched Question 2 Size:", batch["q2"].shape)
print("Pooling Mask 1 Size:", batch["pool_mask1"].shape)
print("Pooling Mask 2 Size:", batch["pool_mask2"].shape)
print("Batched Labels Size:", batch["label"].shape)

Batched Question 1 Size: torch.Size([128, 64])
Batched Question 2 Size: torch.Size([128, 64])
Pooling Mask 1 Size: torch.Size([128, 64])
Pooling Mask 2 Size: torch.Size([128, 64])
Batched Labels Size: torch.Size([128])


In [37]:
# worker
class AdditiveCrossAttentionHead(nn.Module):
    def __init__(
        self,
        hidden_dim,
        value_dim,
        score_dim=model_cfg.ATTENTION_SCORE_DIM,
        dropout=0.0
    ):
        super().__init__()

        self.query_proj = nn.Linear(hidden_dim, score_dim, bias=False)
        self.key_proj = nn.Linear(hidden_dim, score_dim, bias=False)
        self.score = nn.Linear(score_dim, 1, bias=False)
        self.value_proj = nn.Linear(hidden_dim, value_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value, query_mask, key_mask):
        q = self.query_proj(query).unsqueeze(2)
        k = self.key_proj(key_value).unsqueeze(1)

        logits = self.score(torch.tanh(q + k)).squeeze(-1)

        logits = logits.masked_fill(
            ~key_mask.unsqueeze(1),
            torch.finfo(logits.dtype).min
        )

        weights = torch.softmax(logits, dim=-1)
        weights = self.dropout(weights)

        aligned = torch.bmm(
            weights,
            self.value_proj(key_value)
        )

        return aligned * query_mask.unsqueeze(-1).to(aligned.dtype)

In [38]:
# worker
class MultiHeadCrossAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        num_heads=model_cfg.NUM_HEADS,
        score_dim=model_cfg.ATTENTION_SCORE_DIM,
        dropout=model_cfg.ATTENTION_DROPOUT
    ):
        super().__init__()

        value_dim = hidden_dim // num_heads

        self.heads = nn.ModuleList([
            AdditiveCrossAttentionHead(
                hidden_dim, value_dim, score_dim, dropout
            )
            for _ in range(num_heads)
        ])

        self.out_linear = nn.Linear(
            value_dim * num_heads,
            hidden_dim
        )

    def forward(self, query, key_value, query_mask, key_mask):
        contexts = [
            head(query, key_value, query_mask, key_mask)
            for head in self.heads
        ]

        x = torch.cat(contexts, dim=-1)
        x = self.out_linear(x)

        return x * query_mask.unsqueeze(-1).to(x.dtype)


In [39]:
# worker
class AttentionHead(nn.Module):
    def __init__(self, hidden_dim, proj_dim, mask_fill_num=model_cfg.MASK_FILL_NUM,
                 dropout=model_cfg.ATTENTION_DROPOUT):
        super().__init__()
        self.W = nn.Linear(hidden_dim, proj_dim)           # project to subspace
        self.V = nn.Linear(proj_dim, 1, bias=False)        # score
        self.mask_fill_num = mask_fill_num
        self.dropout = nn.Dropout(dropout)

    def forward(self, lstm_output, mask):
        proj = self.W(lstm_output)                         # [B, L, proj_dim]
        energy = torch.tanh(proj)
        scores = self.V(energy).squeeze(-1)                # [B, L]
        scores = scores.masked_fill(
            mask == 0, 
            torch.finfo(scores.dtype).min
        )
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        # Pool from the projected features (not original lstm_output)
        masked_proj = proj * mask.unsqueeze(-1)
        context = torch.bmm(weights.unsqueeze(1), masked_proj).squeeze(1)   # [B, proj_dim]
        return context

In [40]:
# worker
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads=model_cfg.NUM_HEADS):
        super().__init__()
        head_dim = hidden_dim // num_heads
        self.heads = nn.ModuleList([
            AttentionHead(hidden_dim, head_dim) for _ in range(num_heads)
        ])
        self.out_linear = nn.Linear(head_dim*num_heads, hidden_dim)

    def forward(self, hidden_state, mask):
        x = torch.cat([h(hidden_state, mask) for h in self.heads], dim=-1)
        x = self.out_linear(x)
        return x

In [41]:
# worker
class QuoraSiameseClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        pad_idx,
        config=model_cfg
    ):
        super().__init__()

        self.config = config
        self.pad_idx = pad_idx

        # Trainable embedding initialized from scratch
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=config.EMB_DIM,
            padding_idx=pad_idx
        )

        self.emb_norm = nn.LayerNorm(config.EMB_DIM)
        self.emb_dropout = nn.Dropout(config.EMB_DP)

        self.LSTM = nn.LSTM(
            input_size=config.EMB_DIM,
            hidden_size=config.HIDDEN_DIM,
            bidirectional=config.BIDIRECTIONAL,
            num_layers=config.NUM_LAYERS,
            dropout=config.DROPOUT if config.NUM_LAYERS > 1 else 0.0,
            batch_first=True
        )

        self.lstm_norm = nn.LayerNorm(config.LSTM_OUT)
        self.cross_attention = MultiHeadCrossAttention(config.LSTM_OUT)
        self.cross_norm = nn.LayerNorm(config.LSTM_OUT)

        self.attention = MultiHeadAttention(
            config.LSTM_OUT
        )

        if config.ATTENTION_PROJECTION:
            self.proj = nn.Linear(
                config.LSTM_OUT,
                config.PROJECT_DIM
            )
        else:
            self.proj = nn.Identity()

        self.attn_norm = nn.LayerNorm(
            config.LSTM_OUT
        )

        self.fc_dims = self._build_fc_layers(
            input_dim=config.INPUT_FC_DIM,
            fc_dims=config.FC_DIMS,
            dropout=config.FC_DP
        )

    def _build_fc_layers(self, input_dim, fc_dims, dropout):
        layers = []

        for dim in fc_dims:
            layers += [
                nn.Linear(input_dim, dim),
                nn.GELU(),
                nn.Dropout(dropout)
            ]
            input_dim = dim

        layers.append(
            nn.Linear(input_dim, 1)
        )

        return nn.Sequential(*layers)

    def _create_mask(self, question):
        return question.ne(self.pad_idx)


    def _encode(self, question):
        mask = self._create_mask(question)
    
        emb = self.embedding(question)
    
        if self.config.LAYER_NORM_EMB:
            emb = self.emb_norm(emb)
    
        emb = self.emb_dropout(emb)
    
        lengths = mask.sum(dim=1).cpu()
    
        packed = pack_padded_sequence(
            emb,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )
    
        encoded, _ = self.LSTM(packed)
    
        out, _ = pad_packed_sequence(
            encoded,
            batch_first=True,
            total_length=question.size(1)
        )

        if self.config.LAYER_NORM_LSTM:
            out = self.lstm_norm(out)
    
        return out, mask

    def _pool(self, hidden, mask, stopword_mask):
        pool_mask = mask & stopword_mask
        pool_mask = torch.where(
            pool_mask.any(dim=1, keepdim=True),
            pool_mask,
            mask
        )
        pooled = self.attention(hidden, pool_mask)
    
        if self.config.LAYER_NORM_ATTENTION:
            pooled = self.attn_norm(pooled)

        return self.proj(pooled)



    def forward(self, q1, q2, pool_mask1, pool_mask2):
        h1, mask1 = self._encode(q1)
        h2, mask2 = self._encode(q2)
    
        # Bidirectional Bahdanau cross-attention
        cross1 = self.cross_attention(h1, h2, mask1, mask2)
        cross2 = self.cross_attention(h2, h1, mask2, mask1)
    
        if self.config.SKIP_CONNECTION:
            h1 = self.cross_norm(h1 + cross1)
            h2 = self.cross_norm(h2 + cross2)
        else:
            h1, h2 = cross1, cross2
    
        # Independent additive Bahdanau final pooling
        v1 = self._pool(h1, mask1, pool_mask1)
        v2 = self._pool(h2, mask2, pool_mask2)
    
        cosine_sim = F.cosine_similarity(v1, v2).unsqueeze(-1)
    
        feat = torch.cat(
            [v1, v2, torch.abs(v1 - v2), v1 * v2, cosine_sim],
            dim=-1
        )
    
        logits = self.fc_dims(feat)
    
        return logits.squeeze(-1)

In [42]:
def export_model_from_notebook(
    notebook_path: str,
    output_path: Path,
    class_names=(
    "AttentionHead",
    "MultiHeadAttention",
    "AdditiveCrossAttentionHead",
    "MultiHeadCrossAttention",
    "QuoraSiameseClassifier",
    "ModelConfig",
    ),
):
    """Export cells containing the requested top-level model classes."""
    import ast

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    target_classes = set(class_names)
    extracted_cells = []

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        defined_classes = {
            node.name
            for node in tree.body
            if isinstance(node, ast.ClassDef)
        }

        matched_classes = defined_classes & target_classes

        if matched_classes:
            extracted_cells.append((source, matched_classes))

    if not extracted_cells:
        raise ValueError(
            "No matching class definitions found in the notebook."
        )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(
            "import inspect\n"
            "import torch\n"
            "import torch.nn as nn\n"
            "import torch.nn.functional as F\n"
            "from torch.nn.utils.rnn import "
            "pack_padded_sequence, pad_packed_sequence\n\n"
        )

        for source, matched_classes in extracted_cells:
            clean_source = "\n".join(
                line.rstrip() for line in source.splitlines()
            )
            f.write(clean_source)
            f.write("\n\n")

            if "ModelConfig" in matched_classes:
                f.write("model_cfg = ModelConfig()\n\n")

    print(
        f">>> Exported {len(extracted_cells)} cell(s) → {output_path}"
    )

In [43]:
export_model_from_notebook(path_cfg.NOTEBOOK_PATH, path_cfg.MODEL_SCRIPT)

>>> Exported 6 cell(s) → /kaggle/working/qqp/02b_stopmask_model_architecture.py


In [44]:
model = QuoraSiameseClassifier(
    vocab_size=tokenizer.vocab_size,
    config=model_cfg,
    pad_idx=tokenizer.pad_idx
)

print(model)

QuoraSiameseClassifier(
  (embedding): Embedding(30000, 100, padding_idx=0)
  (emb_norm): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
  (emb_dropout): Dropout(p=0.2, inplace=False)
  (LSTM): LSTM(100, 384, num_layers=2, batch_first=True, dropout=0.35, bidirectional=True)
  (lstm_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (cross_attention): MultiHeadCrossAttention(
    (heads): ModuleList(
      (0-3): 4 x AdditiveCrossAttentionHead(
        (query_proj): Linear(in_features=768, out_features=64, bias=False)
        (key_proj): Linear(in_features=768, out_features=64, bias=False)
        (score): Linear(in_features=64, out_features=1, bias=False)
        (value_proj): Linear(in_features=768, out_features=192, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (out_linear): Linear(in_features=768, out_features=768, bias=True)
  )
  (cross_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attention): MultiHeadAttention

In [45]:
model.eval()

with torch.no_grad():
    logits = model(
        batch["q1"][:2],
        batch["q2"][:2],
        batch["pool_mask1"][:2],
        batch["pool_mask2"][:2]
    )

print("Output shape:", logits.shape)
print("Finite predictions:", torch.isfinite(logits).all().item())
print("Classifier input:", model.fc_dims[0].in_features)

Output shape: torch.Size([2])
Finite predictions: True
Classifier input: 3073


In [46]:
print(f"Total Number of Parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Trainable Number of Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Total Number of Parameters: 14209161
Trainable Number of Parameters: 14209161


In [47]:
# worker
class EarlyStopping:
    def __init__(self, patience=5, mode="min", min_delta=0.0):
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta

        self.should_stop = False
        self.best_score = None
        self.counter = 0

    def step(self, current_score):
        if self.best_score is None:
            self.best_score = current_score
            return True

        if self.mode == "min":
            improvement = self.best_score - current_score > self.min_delta
        else:
            improvement = current_score - self.best_score > self.min_delta

        if improvement:
            self.best_score = current_score
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False

In [48]:
# worker
class TrainingHistory:
    def __init__(self):
        self.history = defaultdict(list)

    def update(self, train_loss, val_loss, train_metrics, val_metrics, optimizer):
        self.history["train_loss"].append(train_loss)
        self.history["val_loss"].append(val_loss)
        for k_t, v_t in train_metrics.items():
            self.history[f"train_{k_t.lower()}"].append(v_t)
        for k_v, v_v in val_metrics.items():
            self.history[f"val_{k_v.lower()}"].append(v_v)
        self.history["lr"].append(optimizer.param_groups[0]["lr"])

    def save(self, path: str):
        """Save training history to a JSON file."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.history, f, ensure_ascii=False, indent=2)

        print(f"Training History saved successfully at {path}")

    @classmethod
    def load(cls, path: str):
        "Load training history from a JSON file."
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"No file Found at {path}")
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        obj = cls()
        obj.history = data

        return obj

In [49]:
# Identify the existing experiment
EXPERIMENT_NAME = (
    f"Quora-Question-Pairs/{model_cfg.MODEL_TYPE}"
)

# The Kaggle artifact location
KAGGLE_ARTIFACT_ROOT = (
    path_cfg.MLFLOW_DIR
    / "Quora-Question-Pairs"
    / model_cfg.MODEL_TYPE
).as_uri()

# Save the original Windows location for later
PATH_MAP_FILE = WORK_ROOT / "mlflow-path-map.json"

with sqlite3.connect(sys_cfg._DB_PATH) as conn:

    experiment = conn.execute(
        """
        SELECT experiment_id, artifact_location
        FROM experiments
        WHERE name = ?
        """,
        (EXPERIMENT_NAME,)
    ).fetchone()

    if experiment is None:
        raise RuntimeError(
            f"Experiment not found in the imported database: "
            f"{EXPERIMENT_NAME}"
        )

    experiment_id, original_location = experiment

    if PATH_MAP_FILE.exists():
        path_map = json.loads(PATH_MAP_FILE.read_text())
        if (
            path_map["experiment_id"] != experiment_id
            or path_map["kaggle_root"] != KAGGLE_ARTIFACT_ROOT
            or original_location != KAGGLE_ARTIFACT_ROOT
        ):
            raise RuntimeError("Unexpected MLflow path state.")
        print(">>> MLflow artifact path already configured.")

    else:
        if original_location == KAGGLE_ARTIFACT_ROOT:
            raise RuntimeError(
                "Experiment already points to Kaggle, "
                "but its original Windows path is unknown."
            )

        path_map = {
            "experiment_id": experiment_id,
            "windows_root": original_location,
            "kaggle_root": KAGGLE_ARTIFACT_ROOT,
        }

        PATH_MAP_FILE.write_text(
            json.dumps(path_map, indent=2),
            encoding="utf-8"
        )

        conn.execute(
            """
            UPDATE experiments
            SET artifact_location = ?
            WHERE experiment_id = ?
            """,
            (KAGGLE_ARTIFACT_ROOT, experiment_id)
        )

        print(">>> MLflow artifact path switched to Kaggle!")

print("Original path:", path_map["windows_root"])
print("Current path:", KAGGLE_ARTIFACT_ROOT)

>>> MLflow artifact path switched to Kaggle!
Original path: file:///C:/Users/98922/Documents/python_scripts/AI/projects/quora-questions-pairs/mlruns/Quora-Question-Pairs/LSTM_attention
Current path: file:///kaggle/working/qqp/mlruns/Quora-Question-Pairs/LSTM_attention


In [50]:
# worker
class MLflowTracker:
    def __init__(self, project_name, run_type, config_dict, mlflow_dir, tracking_uri=None):
        model_type = config_dict["model"]["model_type"]
        attention_type = config_dict["model"]["attention_type"]
        tokenizer_type = config_dict["model"]["tokenizer_type"]
        embedding_type = config_dict["model"]["embedding_type"]
        self.experiment_name = f"{project_name}/{model_type}"
        self.attn_type = attention_type
        self.model_type = model_type
        self.tokenizer_type = tokenizer_type
        self.embedding_type = embedding_type
        
        self.base_run_name = (
            f"{model_type}-{attention_type}-{tokenizer_type}"
        )
        self.run_type = run_type
        self.config_dict = config_dict
        if tracking_uri:
            mlflow.set_tracking_uri(tracking_uri)
        else:
            mlflow.set_tracking_uri("http://localhost:5000")

        self.experiment = mlflow.get_experiment_by_name(self.experiment_name)
        artifact_dir = (mlflow_dir / project_name / model_type).as_uri()
        if self.experiment is None:
            experiment_id = mlflow.create_experiment(
                name=self.experiment_name,
                artifact_location=artifact_dir
            )
            self.experiment = mlflow.get_experiment(experiment_id)
            print(f">>> Created new experiment: {self.experiment_name}")
        else:
            print(f">>> Using existing experiment: {self.experiment_name}")

        mlflow.set_experiment(self.experiment_name)

        self.run_name = self._generated_versioned_run_name()
        print(f">>> Run Name: {self.run_name}")

    def _generated_versioned_run_name(self):
        base = self.base_run_name
        experiment = self.experiment
        if self.experiment is None:
            return f"{base}-v1"
    
        runs_df = mlflow.search_runs(
            experiment_ids=[experiment.experiment_id],
            filter_string="tags.status = 'completed'"
        )
        if runs_df.empty:
            return f"{base}-v1"
    
        prefix = f"{base}-v"
        mask = runs_df["tags.mlflow.runName"].str.startswith(prefix, na=False)
        matching = runs_df.loc[mask, "tags.mlflow.runName"]
    
        if matching.empty:
            return f"{base}-v1"
    
        versions = []
        for name in matching:
            try:
                version_str = name.split("-v")[-1]
                versions.append(int(version_str))
            except (ValueError, IndexError):
                continue
        next_version = max(versions) + 1 if versions else 1
        return f"{base}-v{next_version}"
            
        
    def start_run(self):
        self.run = mlflow.start_run(run_name=self.run_name)
        self.run_id = self.run.info.run_id
        return self.run

    def log_param(self, param_name, param):
        mlflow.log_param(param_name, param)
    def log_params(self, params):
        mlflow.log_params(params)
    def log_metric(self, name, value, epoch):
        mlflow.log_metric(name, value, step=epoch)
    def log_config_params(self):
        for key, value in self.config_dict.items():
            if key not in ["system", "path"]:
                mlflow.log_params(value) 
        
    def _log_losses(self, train_loss, val_loss, epoch):
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
    def _log_metrics(self, metrics_dict, epoch, prefix):
        for metric, value in metrics_dict.items():
            mlflow.log_metric(f"{prefix}_{metric.lower()}", value, step=epoch)
    def log_epoch(self, epoch, train_loss, val_loss, train_results, val_results, lr):
        self._log_losses(train_loss, val_loss, epoch)
        self._log_metrics(train_results, epoch, prefix="train")
        self._log_metrics(val_results, epoch, prefix="val")
        mlflow.log_metric("learning_rate", lr, step=epoch)
    def save_state_dict(self, state_dict, checkpoint_path):
        mlflow.pytorch.save_state_dict(
            state_dict,
            path=checkpoint_path
        )

    @staticmethod
    def load_state_dict(model, checkpoint_path, device):
        loaded_state_dict = mlflow.pytorch.load_state_dict(
            str(checkpoint_path),
            map_location=device
        )
    
        model.load_state_dict(loaded_state_dict)
    
        return model
        
    def log_best_model(self, model):
        mlflow_logger = logging.getLogger("mlflow.pytorch")
        original_level = mlflow_logger.level
        mlflow_logger.setLevel(logging.ERROR)

        mlflow.pytorch.log_model(
            model,
            name="best_models",
            registered_model_name=self.run_name,
            serialization_format="pickle",
        )

        mlflow_logger.setLevel(original_level)
    def log_artifact(self, artifact_path):
        mlflow.log_artifact(artifact_path)

    def log_artifact_folder(self, artifact_folder):
        mlflow.log_artifacts(artifact_folder)

    def build_run_summary(self, best_threshold, training_metrics, calibrated_metrics, total_time, avg_time_per_epoch):
        summary = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "run_type": self.run_type,
            "model_type": self.model_type,
            "attention_type": self.attn_type,
            "tokenizer_type": self.tokenizer_type,
            "embedding_type": self.embedding_type,
            "run_id": self.run_id,
            "best_metrics_in_training": training_metrics,
            "best_threshold": best_threshold,
            "best_calibrated_metrics": calibrated_metrics,
            "total_training_time_in_min": total_time,
            "average_training_time_per_epoch": avg_time_per_epoch,
            "params": {}
        }
        for k, v in self.config_dict.items():
            if k not in ["system", "path"]:
                summary["params"][k] = v

        return summary
    def log_summary(self, summary, artifact_name="run_sammary.json"):
        mlflow.log_dict(summary, artifact_name)

    def log_history(self, history, artifact_name="training_history.json"):
        mlflow.log_dict(history, artifact_name)

    def set_final_tags(
        self,
        best_score,
        best_calib_score,
        best_threshold,
        total_training_time,
        world_size,
        gpu_hours,
        batch_size_per_replica
    ):
        mlflow.set_tags({
            "status": "completed",
            "model_type": self.model_type,
            "attention_type": self.attn_type,
            "tokenizer_type": self.tokenizer_type,
            "embedding_type": self.embedding_type,
            "best_training_score": best_score,
            "best_calibrated_score": best_calib_score,
            "best_threshold": best_threshold,
            "total_training_time_in_min": total_training_time,
            "world_size": world_size,
            "gpu_hours": gpu_hours,
            "batch_size_per_replica": batch_size_per_replica,
        })

In [57]:
# worker
class Trainer:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        device,
        history,
        mlflow_tracker,
        config=train_cfg,
        scaler=None
    ):
        self.model = model
        self.base_model = (
            model.module if sys_cfg.USE_DDP else model
        )
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.history = history
        self.config = config
        self.criterion = criterion
        self.criterion_eval = nn.BCEWithLogitsLoss()
        self.optimizer = optimizer
        self.tracker = mlflow_tracker
        self.scaler = scaler
        
        self.early_stopper = EarlyStopping(
            patience=config.EARLY_STOP_PATIENCE,
            mode=config.EARLY_STOP_MODE,
            min_delta=config.EARLY_STOP_MIN_DELTA
        )
        self.scheduler = self._create_scheduler(config.SCHEDULER_TYPE)

        threshold = config.METRICS_THRESHOLD
        self.best_threshold = threshold
        metrics_cls = [BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score]
        self.train_metrics = {m.__name__.replace("Binary", ""): m(threshold).to(device) for m in metrics_cls}
        self.train_metrics["AUROC"] = AUROC(task="binary").to(device)
        self.train_metrics["AveragePrecision"] = AveragePrecision(task="binary").to(device)
        self.val_metrics = {m.__name__.replace("Binary", ""): m(threshold).to(device) for m in metrics_cls}
        self.val_metrics['AUROC'] = AUROC(task="binary").to(device)
        self.val_metrics['AveragePrecision'] = AveragePrecision(task="binary").to(device)
        self.pr_curve = PrecisionRecallCurve(task="binary").to(device)

        self.best_checkpoint_score = float('-inf') if config.CHECKPOINT_MODE == 'max' else float('inf')
        self.checkpoint_mode = config.CHECKPOINT_MODE

        self.optimizer_steps = 0
        self.samples_processed = 0
        self.training_seconds = 0.0
        self.epoch_durations = []

        self.current_epoch = 0

        
    def _create_scheduler(self, scheduler_type):
        if scheduler_type == "CosineAnnealing":
            return CosineAnnealingLR(
                optimizer=self.optimizer,
                T_max=self.config.EPOCHS,
                eta_min=self.config.SCHEDULER_ETA_MIN
            )
        elif scheduler_type == "ReduceLROnPlateau":
            return ReduceLROnPlateau(
                optimizer=self.optimizer,
                patience=self.config.SCHEDULER_PATIENCE,
                factor=self.config.SCHEDULER_FACTOR,
                mode=self.config.SCHEDULER_MODE,
                min_lr=self.config.SCHEDULER_MIN_LR,
                threshold=self.config.SCHEDULER_THRESHOLD,
                threshold_mode=self.config.SCHEDULER_THRESHOLD_MODE
            )
        elif scheduler_type == "OneCycleLR":
            return OneCycleLR(
                optimizer=self.optimizer,
                max_lr=self.config.LEARNING_RATE,
                epochs=self.config.EPOCHS,
                steps_per_epoch=len(self.train_loader),
                pct_start=self.config.SCHEDULER_PCT_START,        
                div_factor=self.config.SCHEDULER_DIV_FACTOR,        
                final_div_factor=self.config.SCHEDULER_FINAL_DIV_FACTOR 
            )
        elif scheduler_type == "CosineAnnealingWarmRestarts":
            return CosineAnnealingWarmRestarts(
                optimizer=self.optimizer,
                T_0=self.config.SCHEDULER_T_0,
                T_mult=self.config.SCHEDULER_T_MULT,
                eta_min=self.config.SCHEDULER_ETA_MIN
            )
        elif scheduler_type in (None, "none"):
            if sys_cfg.IS_MAIN:
                print("Warning: No scheduler")
            return None
        else:
            raise ValueError(f"Unknown scheduler: {scheduler_type}")

    def _check_scheduler(self, scheduler_value):
        if self.scheduler:
            if isinstance(self.scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                old_lr = self.optimizer.param_groups[0]["lr"]
                self.scheduler.step(scheduler_value)
                new_lr = self.optimizer.param_groups[0]["lr"]
                if new_lr < old_lr and sys_cfg.IS_MAIN:
                    print(f">>> LR reduced: {old_lr:.6f} → {new_lr:.6f}")
            else:
                self.scheduler.step()
    
    def _reset_metrics(self, metrics_dict):
        for metric in metrics_dict.values():
            metric.reset()
    
    def _update_metrics(self, metrics_dict, probs, labels):
        for metric in metrics_dict.values():
            metric.update(probs, labels)
    def _update_threshold(self, metrics_dict):
        for metric in metrics_dict.values():
            if hasattr(metric, "threshold"):
                metric.threshold = self.best_threshold
    
    def _compute_metrics(self, metrics_dict):
        return {k:m.compute().item() for k, m in metrics_dict.items()}

    def _get_metric_value(self, metric_name, val_loss, val_results):
        if metric_name == "loss":
            return val_loss
        else:
            return val_results[metric_name]

    def _is_checkpoint_better(self, current_value: float) -> bool:
        if self.checkpoint_mode == 'max':
            return current_value > self.best_checkpoint_score
        else:
            return current_value < self.best_checkpoint_score

    def _reduce_loss(self, total_loss, num_samples):
        if sys_cfg.USE_DDP:
            stats = torch.tensor(
                [total_loss, num_samples],
                dtype=torch.float64,
                device=self.device
            )
    
            dist.all_reduce(stats, op=dist.ReduceOp.SUM)
    
            total_loss = stats[0].item()
            num_samples = int(stats[1].item())
    
        return total_loss / num_samples, num_samples
    
    def _backprop_with_scaler(self, q1, q2, labels, pool_mask1, pool_mask2):
        device_type = "cuda" if "cuda" in str(self.device) else "cpu"
        with autocast(enabled=self.scaler is not None, device_type=device_type):
            logits = self.model(q1, q2, pool_mask1, pool_mask2)
            loss = self.criterion(logits, labels)
        if self.scaler is not None:
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)

            if self.config.CLIP_NORM is not None and self.config.CLIP_NORM > 0:
                clip_grad_norm_(self.model.parameters(), self.config.CLIP_NORM)

            previous_scale = self.scaler.get_scale()
            self.scaler.step(self.optimizer)
            self.scaler.update()

            if self.scaler.get_scale() >= previous_scale:
                self.optimizer_steps += 1
        else:
            loss.backward()
            if self.config.CLIP_NORM is not None and self.config.CLIP_NORM > 0:
                clip_grad_norm_(self.model.parameters(), self.config.CLIP_NORM)
                
            self.optimizer.step()
            self.optimizer_steps += 1

        return loss, logits
            
    def _progress(self, loader, desc):
        if not sys_cfg.IS_MAIN:
            yield from loader
            return

        total = len(loader)
        print("@@QQP_PROGRESS@@" + json.dumps({
            "event": "start", "desc": desc, "total": total
        }), flush=True)

        last_update = perf_counter()
        for step, batch in enumerate(loader, start=1):
            yield batch
            now = perf_counter()
            if now - last_update >= 0.1 or step == total:
                print("@@QQP_PROGRESS@@" + json.dumps({
                    "event": "update", "step": step
                }), flush=True)
                last_update = now

        print("@@QQP_PROGRESS@@" + json.dumps({
            "event": "end"
        }), flush=True)

    def train_one_epoch(self):
        self.model.train()
        if sys_cfg.USE_DDP:
            self.train_loader.sampler.set_epoch(self.current_epoch)
        self._update_threshold(self.train_metrics)
        self._reset_metrics(self.train_metrics)
        total_loss = 0.0
        num_samples = 0

        for batch in self._progress(self.train_loader, "Train"):
            q1 = batch["q1"].to(self.device)
            q2 = batch["q2"].to(self.device)
            labels = batch["label"].to(self.device).long()

            pool_mask1 = batch["pool_mask1"].to(self.device)
            pool_mask2 = batch["pool_mask2"].to(self.device)

            self.optimizer.zero_grad()
            
            loss, logits = self._backprop_with_scaler(
                q1, q2, labels, pool_mask1, pool_mask2
            )
            
            total_loss += (loss.item() * q1.size(0))
            num_samples += q1.size(0)

            probs = torch.sigmoid(logits)
            self._update_metrics(self.train_metrics, probs, labels)

        total_loss, global_samples = self._reduce_loss(
            total_loss, num_samples
        )
        
        self.samples_processed += global_samples
        results = self._compute_metrics(self.train_metrics)
        return total_loss, results

    @torch.no_grad()
    def evaluate(self, use_optimal_threshold=True):
        self.model.eval()
        total_loss = 0.0
        num_samples = 0
        all_logits, all_labels = [], []
    
        for batch in self._progress(self.val_loader, "Validation"):
            q1 = batch["q1"].to(self.device)
            q2 = batch["q2"].to(self.device)
            labels = batch["label"].to(self.device).long()

            pool_mask1 = batch["pool_mask1"].to(self.device)
            pool_mask2 = batch["pool_mask2"].to(self.device)
    
            logits = self.base_model(
                q1, q2, pool_mask1, pool_mask2
            )
            loss = self.criterion_eval(logits, labels.float())
            total_loss += loss.item() * q1.size(0)
            num_samples += q1.size(0)
    
            all_logits.append(logits)
            all_labels.append(labels)
    
        total_loss, _ = self._reduce_loss(
            total_loss, num_samples
        )
    
        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels)
        all_probs = torch.sigmoid(all_logits)
    
        # Find optimal threshold using self.pr_curve (if use_optimal_threshold)
        if use_optimal_threshold:
            self.pr_curve.reset()
            all_labels = all_labels.long()
            self.pr_curve.update(all_probs, all_labels)
            precision, recall, thresholds = self.pr_curve.compute()
            # Ensure thresholds is not empty
            if thresholds.numel() > 0:
                f1_scores = (
                    2 * precision[:-1] * recall[:-1]
                    / (precision[:-1] + recall[:-1] + 1e-8)
                )
                best_idx = torch.argmax(f1_scores)
                self.best_threshold = thresholds[best_idx].item()
            else:
                self.best_threshold = self.config.METRICS_THRESHOLD
        else:
            self.best_threshold = self.config.METRICS_THRESHOLD
    
        self._update_threshold(self.val_metrics)
    
        # Reset, update, compute using existing helpers
        self._reset_metrics(self.val_metrics)
        self._update_metrics(self.val_metrics, all_probs, all_labels)
        results = self._compute_metrics(self.val_metrics)
    
        return total_loss, results
    def log_one_epoch(self, train_loss, train_results, val_loss, val_results, best_thresh, lr):
        print(
            f"Training Results:\n\tLoss --> {train_loss:.4f}"
        )
        train_string = ""
        val_string = ""
        for k, v in train_results.items():
            train_result = f"{k} --> {v:.4f} | "
            train_string += train_result
        print(f"\t{train_string}")
        print(f"\tLearning Rate --> {lr:.4f}\n")
        print(
            f"Validation Results:\n\tLoss --> {val_loss:.4f}"
            )
        print(f"\tOptimal Best Threhsold --> {best_thresh}")
        for k, v in val_results.items():
            val_result = f"{k} --> {v:.4f} | "
            val_string += val_result
        print(f"\t{val_string}")

    @torch.no_grad()
    def find_optimal_threshold(self):
        self.model.eval()
        _, results = self.evaluate(use_optimal_threshold=True)
        best_threshold = self.best_threshold
    
        if sys_cfg.IS_MAIN:
            print(f">>> Optimal threshold: {best_threshold:.4f}")
            print(f">>> At that threshold --> F1: {results['F1Score']:.4f}, Precision: {results['Precision']:.4f}, Recall: {results['Recall']:.4f}")
        return best_threshold, results
    
    def fit(self, num_epochs, config=path_cfg):
        if self.device.type == "cuda":
            torch.cuda.synchronize(self.device)
            torch.cuda.reset_peak_memory_stats(self.device)
        
        if sys_cfg.USE_DDP:
            dist.barrier()
        
        train_start = perf_counter()
        
        if sys_cfg.IS_MAIN:
            print(">>> Training Started...")
        
        with (
            self.tracker.start_run()
            if sys_cfg.IS_MAIN else nullcontext()
        ):
            if sys_cfg.IS_MAIN:
                self.tracker.log_config_params()
            
                self.tracker.log_artifact_folder(config.TOKENIZER_DIR)
            
                self.tracker.log_artifact(config.LABEL_MAPPING_PATH)
                self.tracker.log_artifact(config.CONFIG_PATH)
                self.tracker.log_artifact(config.MODEL_SCRIPT)
                self.tracker.log_artifact(config.REQ_SCRIPT)
                self.tracker.log_artifact(config.REQ_TEXT)
                self.tracker.log_artifact(config.STOPWORDS_PATH)
                execution_info = {
                    "training_strategy": sys_cfg.TRAINING_STRATEGY,
                    "gpu_count": sys_cfg.GPU_COUNT,
                    "available_gpu_count": sys_cfg.AVAILABLE_GPU_COUNT,
                    "gpu_type": sys_cfg.GPU_TYPE,
                    "world_size": sys_cfg.WORLD_SIZE
                }
                
                self.tracker.log_params(execution_info)

            best_metrics = {}
            
            for epoch in range(num_epochs):
                self.current_epoch = epoch
                if sys_cfg.IS_MAIN:
                    print(f"Epoch {epoch+1}/{num_epochs}")
                
                epoch_start = perf_counter()
                training_start = perf_counter()
                
                train_loss, train_results = self.train_one_epoch()
                
                self.training_seconds += perf_counter() - training_start
                
                val_loss, val_results = self.evaluate()

                early_stop_value = self._get_metric_value(
                    self.config.EARLY_STOP_METRIC, val_loss, val_results
                )
                scheduler_value = self._get_metric_value(
                    self.config.SCHEDULER_METRIC, val_loss, val_results
                )
                checkpoint_value = self._get_metric_value(
                    self.config.CHECKPOINT_METRIC, val_loss, val_results
                )
        
                self._check_scheduler(scheduler_value)

                if sys_cfg.IS_MAIN:
                    self.history.update(
                        train_loss=train_loss,
                        val_loss=val_loss,
                        train_metrics=train_results,
                        val_metrics=val_results,
                        optimizer=self.optimizer
                    )
                    self.tracker.log_history(dict(self.history.history))
                    lr = self.optimizer.param_groups[0]["lr"]
                    self.log_one_epoch(
                        train_loss=train_loss,
                        train_results=train_results,
                        val_loss=val_loss,
                        val_results=val_results,
                        best_thresh=self.best_threshold,
                        lr=lr
                    )
                    self.tracker.log_epoch(
                        epoch=epoch,
                        train_loss=train_loss,
                        val_loss=val_loss,
                        train_results=train_results,
                        val_results=val_results,
                        lr=lr
                    )
                    self.tracker.log_metric("best_threshold", self.best_threshold, epoch)
                
                self.early_stopper.step(early_stop_value)
                
                if self._is_checkpoint_better(checkpoint_value):
                    self.best_checkpoint_score = checkpoint_value
                
                    best_metrics = val_results
                
                    if sys_cfg.IS_MAIN:
                        self.tracker.save_state_dict(
                            self.base_model.state_dict(),
                            config.CHECKPOINT_DIR
                        )
                
                        print(
                            f">>> Best model saved! "
                            f"({self.config.CHECKPOINT_METRIC} "
                            f"--> {checkpoint_value:.4f})"
                        )
                    
                if sys_cfg.IS_MAIN:
                    epoch_duration = perf_counter() - epoch_start
                    self.epoch_durations.append(epoch_duration)
                
                if self.early_stopper.should_stop:
                    if sys_cfg.IS_MAIN:
                        print(
                            f">>> Early stopping triggered at Epoch {epoch+1}"
                        )
                    break
                
                if sys_cfg.IS_MAIN:
                    print("=" * sys_cfg.NEXT_LINE_COUNTER)


            if self.device.type == "cuda":
                torch.cuda.synchronize(self.device)
            
            if sys_cfg.USE_DDP:
                dist.barrier()
            
            total_training_time = perf_counter() - train_start
            if sys_cfg.IS_MAIN:
                avg_time_per_epoch = (
                    sum(self.epoch_durations) / len(self.epoch_durations)
                )
            
                total_training_time_in_min = round(
                    total_training_time / 60, 2
                )
            
                avg_time_per_epoch_in_min = round(
                    avg_time_per_epoch / 60, 2
                )
            peak_memory = torch.tensor(
                torch.cuda.max_memory_allocated(self.device) / 1024**2
                if self.device.type == "cuda" else 0.0,
                device=self.device
            )
                
            if sys_cfg.USE_DDP:
                dist.all_reduce(peak_memory, op=dist.ReduceOp.MAX)

            if sys_cfg.IS_MAIN:
                execution_results = {
                    "training_time_seconds": round(total_training_time, 2),
            
                    "gpu_hours": round(
                        sys_cfg.GPU_COUNT * total_training_time / 3600, 4
                    ),
            
                    "optimizer_steps": self.optimizer_steps,
            
                    "samples_processed": self.samples_processed,
            
                    "samples_per_second": round(
                        self.samples_processed / self.training_seconds, 2
                    ),
            
                    "peak_gpu_memory_mb": round(peak_memory.item(), 2)
                }
            
                self.tracker.log_params(execution_results)

            MLflowTracker.load_state_dict(
                self.base_model,
                config.CHECKPOINT_DIR,
                self.device
            )
            self.model.eval()
            final_best_threshold, calibrated_results = self.find_optimal_threshold()
            if sys_cfg.IS_MAIN:
                self.tracker.log_artifact_folder(config.CHECKPOINT_DIR)
                self.tracker.log_best_model(self.base_model)
                print(">>> The Best Model registered at MLflow successfully!")
                summary = self.tracker.build_run_summary(
                    best_threshold=final_best_threshold,
                    training_metrics=best_metrics,
                    calibrated_metrics=calibrated_results,
                    total_time=total_training_time_in_min,
                    avg_time_per_epoch=avg_time_per_epoch_in_min
                )
                summary["execution"] = {
                    **execution_info,
                    **execution_results
                }
                self.tracker.log_param(
                    f"best_training_{self.config.CHECKPOINT_METRIC}", 
                    self.best_checkpoint_score
                )
                self.tracker.log_param(
                    f"best_calibrated_{self.config.CHECKPOINT_METRIC}", 
                    calibrated_results[self.config.CHECKPOINT_METRIC]
                )
                self.tracker.log_param("best_threshold", final_best_threshold)
                self.tracker.log_params(calibrated_results)
                self.tracker.log_param("total_training_time_in_min", total_training_time_in_min)
                self.tracker.log_param("average_time_per_epoch_in_min", avg_time_per_epoch_in_min)
                self.tracker.log_summary(summary)
                self.tracker.log_history(dict(self.history.history))
                self.tracker.log_artifact(config.NOTEBOOK_PATH)
                self.tracker.set_final_tags(
                    best_score=self.best_checkpoint_score,
                
                    best_calib_score=calibrated_results[
                        self.config.CHECKPOINT_METRIC
                    ],
                
                    best_threshold=final_best_threshold,
                    total_training_time=total_training_time_in_min,
                
                    world_size=sys_cfg.WORLD_SIZE,
                    gpu_hours=execution_results["gpu_hours"],
                    batch_size_per_replica=loader_cfg.LOCAL_BATCH_SIZE
                )

In [58]:
# worker
class BCEWithLabelSmoothing(nn.Module):
    def __init__(self,  epsilon, reduction="mean"):
        super().__init__()
        self.epsilon = epsilon
        self.criterion = nn.BCEWithLogitsLoss(reduction=reduction)

    def forward(self, logits, labels):
        labels = labels.float()
        labels = labels * (1 - self.epsilon) + (1 - labels) * self.epsilon
        return self.criterion(logits, labels)

In [59]:
# worker
def main():
    try:
        sys_cfg.configure()
        loader_cfg.configure(sys_cfg.WORLD_SIZE)

        seed_everything(
            sys_cfg.SEED + sys_cfg.RANK,
            deterministic=sys_cfg.IS_DETERMINISTIC
        )

        scaler = set_scaler(sys_cfg)

        # Load data
        train_df = pd.read_csv(path_cfg.TRAIN_CSV_PATH)
        valid_df = pd.read_csv(path_cfg.VALID_CSV_PATH)

        train_df_preprc = preprocess_df(train_df)
        valid_df_preprc = preprocess_df(valid_df)

        # Tokenizer
        stop_words = json.loads(
            path_cfg.STOPWORDS_PATH.read_text(
                encoding="utf-8"
            )
        )

        tokenizer = QuoraTokenizer(
            tokenizer_dir=path_cfg.TOKENIZER_DIR,
            stop_words=stop_words,
            config=token_cfg
        )

        # Datasets
        train_dataset = QuoraDataset(
            train_df_preprc,
            tokenizer=tokenizer
        )

        val_dataset = QuoraDataset(
            valid_df_preprc,
            tokenizer=tokenizer
        )

        # DataLoaders
        train_dataloader = build_dataloader(
            train_dataset,
            is_train=True
        )

        val_dataloader = build_dataloader(
            val_dataset,
            is_train=False
        )

        # Model
        model = QuoraSiameseClassifier(
            vocab_size=tokenizer.vocab_size,
            config=model_cfg,
            pad_idx=tokenizer.pad_idx
        ).to(sys_cfg.DEVICE)

        if sys_cfg.USE_DDP:
            model = DDP(
                model,
                device_ids=[sys_cfg.LOCAL_RANK],
                find_unused_parameters=True
            )

        # MLflow
        mlflow_tracker = None

        if sys_cfg.IS_MAIN:
            configs = configs_dict(path_cfg.CONFIG_PATH)

            tokenizer.save_label_mapping(
                path_cfg.LABEL_MAPPING_PATH
            )

            mlflow_tracker = MLflowTracker(
                project_name="Quora-Question-Pairs",
                run_type="exploring-best-architecture",
                config_dict=configs,
                mlflow_dir=path_cfg.MLFLOW_DIR,
                tracking_uri=sys_cfg.MLFLOW_TRACKING_URI
            )

        history = TrainingHistory()

        criterion = BCEWithLabelSmoothing(
            epsilon=model_cfg.LABEL_SMOOTHING
        )

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=train_cfg.LEARNING_RATE,
            weight_decay=train_cfg.WEIGHT_DECAY
        )

        # Trainer
        trainer = Trainer(
            model=model,
            train_loader=train_dataloader,
            val_loader=val_dataloader,
            criterion=criterion,
            optimizer=optimizer,
            device=sys_cfg.DEVICE,
            history=history,
            config=train_cfg,
            mlflow_tracker=mlflow_tracker,
            scaler=scaler
        )

        # Training
        trainer.fit(
            num_epochs=train_cfg.EPOCHS,
            config=path_cfg
        )

        if sys_cfg.IS_MAIN:
            print("MLflow directory:", path_cfg.MLFLOW_DIR)

            print(
                "Experiment artifact URI:",
                mlflow_tracker.experiment.artifact_location
            )

            print(
                "Current run artifact URI:",
                mlflow.get_run(
                    mlflow_tracker.run_id
                ).info.artifact_uri
            )

    finally:
        if dist.is_initialized():
            dist.destroy_process_group()


In [60]:
notebook = json.loads(
    path_cfg.NOTEBOOK_PATH.read_text(encoding="utf-8")
)

worker_cells = [
    "".join(cell["source"])
    for cell in notebook["cells"]
    if cell["cell_type"] == "code"
    and "".join(cell["source"]).lstrip().startswith("# worker\n")
]

worker_source = "\n\n".join(worker_cells)

worker_source += (
    '\n\nif __name__ == "__main__":\n'
    '    main()\n'
)

WORKER_PATH = WORK_ROOT / "train_worker.py"

compile(worker_source, str(WORKER_PATH), "exec")

WORKER_PATH.write_text(
    worker_source,
    encoding="utf-8"
)

62719

In [61]:
source = WORKER_PATH.read_text(encoding="utf-8")

print("Graphical progress:", "def _progress(" in source)
print("NCCL device fix:", "device_id=cls.DEVICE" in source)
print("Progress messages:", "@@QQP_PROGRESS@@" in source)

Graphical progress: True
NCCL device fix: True
Progress messages: True


In [62]:
num_gpus = torch.cuda.device_count()

replicas = max(1, num_gpus)

if loader_cfg.BATCH_SIZE_STRATEGY == "fixed_global":
    replicas = max(
        n for n in range(1, replicas + 1)
        if loader_cfg.BATCH_SIZE % n == 0
    )

if replicas < num_gpus:
    print(f">>> Using {replicas}/{num_gpus} available GPUs.")

if replicas > 1:
    command = [
        sys.executable,
        "-m",
        "torch.distributed.run",
        "--standalone",
        "--nnodes=1",
        f"--nproc-per-node={replicas}",
        str(WORKER_PATH)
    ]
else:
    command = [
        sys.executable,
        str(WORKER_PATH)
    ]

In [63]:
# Match torchrun's default without printing its OMP setup notice.
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
if replicas > 1:
    env.setdefault("OMP_NUM_THREADS", "1")

progress_bar = None
progress_prefix = "@@QQP_PROGRESS@@"

with subprocess.Popen(
    command,
    cwd=WORK_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    env=env
) as process:
    try:
        for line in process.stdout:
            if not line.startswith(progress_prefix):
                print(line, end="", flush=True)
                continue

            event = json.loads(line[len(progress_prefix):])

            if event["event"] == "start":
                progress_bar = tqdm(
                    total=event["total"],
                    desc=event["desc"],
                    leave=True
                )
            elif event["event"] == "update":
                progress_bar.update(event["step"] - progress_bar.n)
            elif event["event"] == "end":
                progress_bar.close()
                progress_bar = None

        return_code = process.wait()
    finally:
        if progress_bar is not None:
            progress_bar.close()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)

>>> Training strategy: DDP
>>> GPUs used: 2
>>> GPU type: Tesla T4
>>> Using PROTOTYPING mode (Faster).
>>> For Reproducibility, Everything seeded with 28!
>>> Scaler is not Used
>>> Preprocessing complete! Removed empty rows: 1
>>> Preprocessing complete! Removed empty rows: 0
>>> WordPiece tokenizer loaded!
>>> Vocabulary size: 30,000
>>> PAD token: [PAD] (0)
>>> UNK token: [UNK] (1)
>>> Configs Saved as JSON File!
>>> Label mapping saved!
>>> Using existing experiment: Quora-Question-Pairs/LSTM_attention
>>> Run Name: LSTM_attention-MultiHead-Bahdanau-Cross-AdditivePool-StopMask-Wordpiece-v1
>>> Training Started...
Epoch 1/50


Train:   0%|          | 0/2843 [00:00<?, ?it/s]

Validation:   0%|          | 0/316 [00:00<?, ?it/s]

Training Results:
	Loss --> 0.5500
	Accuracy --> 0.7468 | Precision --> 0.6813 | Recall --> 0.5902 | F1Score --> 0.6325 | AUROC --> 0.8095 | AveragePrecision --> 0.7064 | 
	Learning Rate --> 0.0003

Validation Results:
	Loss --> 0.4634
	Optimal Best Threhsold --> 0.5102353692054749
	Accuracy --> 0.7789 | Precision --> 0.6659 | Recall --> 0.8049 | F1Score --> 0.7288 | AUROC --> 0.8639 | AveragePrecision --> 0.7847 | 
>>> Best model saved! (F1Score --> 0.7288)
Epoch 2/50


Train:   0%|          | 0/2843 [00:00<?, ?it/s]

Validation:   0%|          | 0/316 [00:00<?, ?it/s]

... (47 outputs left out)

Validation:   0%|          | 0/316 [00:00<?, ?it/s]

Training Results:
	Loss --> 0.2790
	Accuracy --> 0.9524 | Precision --> 0.9517 | Recall --> 0.9176 | F1Score --> 0.9344 | AUROC --> 0.9879 | AveragePrecision --> 0.9788 | 
	Learning Rate --> 0.0001

Validation Results:
	Loss --> 0.3668
	Optimal Best Threhsold --> 0.6666874289512634
	Accuracy --> 0.8657 | Precision --> 0.7876 | Recall --> 0.8712 | F1Score --> 0.8273 | AUROC --> 0.9171 | AveragePrecision --> 0.8438 | 
>>> Best model saved! (F1Score --> 0.8273)
>>> Early stopping triggered at Epoch 18


Validation:   0%|          | 0/316 [00:00<?, ?it/s]

>>> Optimal threshold: 0.6667
>>> At that threshold --> F1: 0.8273, Precision: 0.7876, Recall: 0.8712
2026/11/27 18:28:43 WARNING mlflow.utils.requirements_utils: Found torch version (2.8.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.8.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/27 18:28:58 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.23.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.23.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
Successfully registered model 'LSTM_attention-MultiHead-Bahdanau-Cross-AdditivePool-StopMask-Wordpiece-v1'.
Created version '1' of m

In [64]:
# Read the saved original and Kaggle paths
path_map = json.loads(
    PATH_MAP_FILE.read_text(encoding="utf-8")
)

windows_root = path_map["windows_root"]
kaggle_root = path_map["kaggle_root"]
experiment_id = path_map["experiment_id"]

# Only rewrite paths belonging to our Kaggle artifact root
def restore_uri(uri):
    if not isinstance(uri, str):
        return uri

    if uri == kaggle_root or uri.startswith(kaggle_root + "/"):
        return windows_root + uri[len(kaggle_root):]

    return uri


with sqlite3.connect(sys_cfg._DB_PATH) as conn:

    # Restore the experiment's original location
    current_root = conn.execute(
        """
        SELECT artifact_location
        FROM experiments
        WHERE experiment_id = ?
        """,
        (experiment_id,)
    ).fetchone()

    if current_root is None or current_root[0] not in (
        windows_root, kaggle_root
    ):
        raise RuntimeError("Unexpected experiment artifact location.")

    conn.execute(
        """
        UPDATE experiments
        SET artifact_location = ?
        WHERE experiment_id = ?
        """,
        (windows_root, experiment_id)
    )

    # Restore paths of newly generated MLflow objects
    fields = [
        ("runs", "artifact_uri"),
        ("logged_models", "artifact_location"),
        ("model_versions", "source"),
        ("model_versions", "storage_location"),
    ]

    for table, column in fields:

        columns = {
            row[1]
            for row in conn.execute(f"PRAGMA table_info({table})")
        }

        if column not in columns:
            continue

        rows = conn.execute(
            f"SELECT rowid, {column} FROM {table}"
        ).fetchall()

        changed = 0

        for row_id, uri in rows:
            restored = restore_uri(uri)

            if restored != uri:
                conn.execute(
                    f"UPDATE {table} SET {column} = ? "
                    f"WHERE rowid = ?",
                    (restored, row_id)
                )
                changed += 1

        print(f">>> {table}.{column}: {changed} paths restored")

print(">>> MLflow database paths restored to Windows!")


# Create a consistent SQLite backup for export
EXPORT_DB = WORK_ROOT / "mlflow-export.db"

with sqlite3.connect(sys_cfg._DB_PATH) as source:
    with sqlite3.connect(EXPORT_DB) as destination:
        source.backup(destination)


# Package everything generated in this Kaggle session
EXPORT_ZIP = Path("/kaggle/working/qqp-transfer.zip")

with zipfile.ZipFile(
    EXPORT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=4
) as archive:

    # Updated database
    archive.write(
        EXPORT_DB,
        arcname=DB_NAME
    )

    # MLflow artifacts, including registered model files
    # No old local mlruns were uploaded, so they are not here.
    for file in path_cfg.MLFLOW_DIR.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(WORK_ROOT).as_posix()
            )

    # Your normal model artifacts/checkpoint
    for file in path_cfg.MODEL_ARTIFACT_DIR.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(WORK_ROOT).as_posix()
            )
    archive.write(
        path_cfg.MODEL_SCRIPT,
        arcname=path_cfg.MODEL_SCRIPT.name
    )

print(f">>> Export ready: {EXPORT_ZIP}")

>>> runs.artifact_uri: 1 paths restored
>>> logged_models.artifact_location: 1 paths restored
>>> model_versions.source: 0 paths restored
>>> model_versions.storage_location: 1 paths restored
>>> MLflow database paths restored to Windows!
>>> Export ready: /kaggle/working/qqp-transfer.zip
